# Clip NatCap Global Layers to the Basilicata MAES Level-2 Grid

This notebook clips two global rasters from the **Natural Capital Alliance (Stanford) Data Hub**
to the **exact pixel grid** (extent, resolution, and CRS) of `LE_D26_basilicata_2017_maesL2.tif`:

1. **Global Soil Erodibility – EPIC K-Factor** (ISRIC SoilGrids-derived USLE K-factor)
2. **GloRESatE – Global Mean Annual Rainfall Erosivity** (~11 km, Das et al. 2024, *Sci Data*)

"Exact" here means the outputs are **snapped to the reference grid**: same CRS (`EPSG:6875`), same
`width`/`height`, same affine `transform`, and therefore pixel-for-pixel alignment with the MAES
Level-2 raster — not just a bounding-box crop. This is done with a `WarpedVRT`
(GDAL's on-the-fly reprojection/resampling), reprojecting straight from the remote COGs via
`/vsicurl/`, so the full multi-GB global rasters never need to be downloaded — only the
Basilicata window is fetched over HTTP range requests.

This mirrors the covariate-extraction pattern used elsewhere in the LandShift / Living Earth
pipeline (NASA DEM, HydroSHEDS, ERA5-Land clipped to `EPSG:6875` for Basilicata), and can serve
as USLE inputs (K-factor × R-factor) alongside the MAES Level-2 land-cover/habitat layer.


In [6]:
import os
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize


## 1. Configuration

In [ ]:
# ── DEMO_MODE ────────────────────────────────────────────────────────────
# True  -> builds small synthetic local rasters standing in for the two NatCap
#          layers (useful offline / for testing the pipeline without network access).
# False -> reads the real datasets directly from the NatCap Data Hub via /vsicurl/
#          (requires outbound HTTPS access to data.naturalcapitalalliance.stanford.edu).
DEMO_MODE = False

# ── Reference raster (defines the exact target grid) ───────────────────────
REFERENCE_TIF = "/Users/gregorygiuliani/Desktop/input/LE_D26_basilicata_2017_maesL2.tif"   # CRS EPSG:6875, 10 m, MAES Level-2

# ── Source datasets (NatCap Data Hub, Cloud-Optimized GeoTIFFs) ────────────
SOURCES = {
    "epic_k_factor": {
        "description": "Global Soil Erodibility - EPIC K-Factor (Sharpley & Williams 1990, "
                        "ISRIC SoilGrids-derived)",
        "url": "https://data.naturalcapitalalliance.stanford.edu/download/global/"
               "erodibility-k-factors/epic-k-factor.tif",
        "resampling": Resampling.bilinear,   # continuous variable -> bilinear
        "nodata_out": np.nan,
    },
    "gloresate_erosivity": {
        "description": "GloRESatE Global Mean Annual Rainfall Erosivity, ~11 km "
                        "(Das et al. 2024, Sci Data, doi:10.5281/zenodo.11078865)",
        "url": "https://data.naturalcapitalalliance.stanford.edu/download/global/"
               "GLORESATE-erosivity/GloRESatE_mean_avg_erosivity.tif",
        "resampling": Resampling.bilinear,   # continuous variable -> bilinear
        "nodata_out": np.nan,
    },
}

OUTPUT_DIR = "clipped_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GDAL/rasterio env tuned for remote COG access via HTTP range requests
GDAL_ENV = dict(
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    CPL_VSIL_CURL_USE_HEAD="NO",
    GDAL_HTTP_MULTIRANGE="YES",
    VSI_CACHE="TRUE",
    GDAL_HTTP_MAX_RETRY="3",
    GDAL_HTTP_RETRY_DELAY="1",
)


## 2. Reference grid

Read the exact target grid (CRS, transform, dimensions, bounds) from the MAES Level-2 raster. Every output will be forced onto this identical grid.

In [ ]:
with rasterio.open(REFERENCE_TIF) as ref:
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height
    ref_bounds = ref.bounds
    ref_res = ref.res

print(f"Reference CRS       : {ref_crs}")
print(f"Reference resolution: {ref_res}")
print(f"Reference dimensions : {ref_width} x {ref_height} (width x height)")
print(f"Reference bounds     : {ref_bounds}")
print(f"Reference transform  :\n{ref_transform}")


## 3. Demo-mode synthetic sources *(skipped when `DEMO_MODE = False`)*

Only runs in `DEMO_MODE`. Builds small local GeoTIFFs in `EPSG:4326` at coarse resolution
(analogous to the real global layers) covering the Basilicata bounding box, so the clipping
logic below can be exercised end-to-end without network access.

In [ ]:
DEMO_SOURCE_PATHS = {}

if DEMO_MODE:
    from rasterio.warp import transform_bounds

    with rasterio.open(REFERENCE_TIF) as ref:
        lon_min, lat_min, lon_max, lat_max = transform_bounds(ref.crs, "EPSG:4326", *ref.bounds)

    # pad a bit so the synthetic global-style raster fully covers the AOI after warping
    pad = 0.5
    lon_min, lat_min = lon_min - pad, lat_min - pad
    lon_max, lat_max = lon_max + pad, lat_max + pad

    demo_specs = {
        "epic_k_factor": dict(res_deg=0.002, seed=1),        # ~SoilGrids-like fine resolution
        "gloresate_erosivity": dict(res_deg=0.10, seed=2),   # ~11 km-like coarse resolution
    }

    for key, spec in demo_specs.items():
        res = spec["res_deg"]
        w = int(np.ceil((lon_max - lon_min) / res))
        h = int(np.ceil((lat_max - lat_min) / res))
        transform = rasterio.transform.from_origin(lon_min, lat_max, res, res)

        rng = np.random.default_rng(spec["seed"])
        yy, xx = np.mgrid[0:h, 0:w]
        # smooth synthetic surface with plausible magnitude, just for pipeline testing
        data = (
            0.02
            + 0.01 * np.sin(xx / 12.0) * np.cos(yy / 15.0)
            + 0.002 * rng.standard_normal((h, w))
        ).astype("float32")

        demo_path = os.path.join(OUTPUT_DIR, f"DEMO_{key}.tif")
        profile = dict(
            driver="GTiff", dtype="float32", count=1, width=w, height=h,
            crs="EPSG:4326", transform=transform, nodata=np.nan,
        )
        with rasterio.open(demo_path, "w", **profile) as dst:
            dst.write(data, 1)

        DEMO_SOURCE_PATHS[key] = demo_path
        print(f"Built synthetic source for '{key}': {demo_path}  ({w} x {h} @ {res} deg)")
else:
    print("DEMO_MODE is False — using the real NatCap Data Hub sources.")


## 4. Perfect-clip function

For a given source raster (local path or `/vsicurl/` remote URL), open it inside a `WarpedVRT`
whose `crs`, `transform`, `width`, and `height` are forced to match the reference exactly. GDAL
then reprojects/resamples on the fly and — for remote COGs — only fetches the bytes needed to
cover that window, via HTTP range requests. The array returned (and the file written) therefore
has **identical shape, transform, and CRS to the reference raster**, guaranteeing perfect
pixel alignment for later stacking/analysis.

In [39]:
def clip_to_reference(source_path_or_url, out_path, resampling, nodata_out,
                       ref_crs=ref_crs, ref_transform=ref_transform,
                       ref_width=ref_width, ref_height=ref_height):
    """Reproject + resample + clip `source_path_or_url` onto the exact reference grid."""
    with rasterio.Env(**GDAL_ENV):
        with rasterio.open(source_path_or_url) as src:
            with WarpedVRT(
                src,
                crs=ref_crs,
                transform=ref_transform,
                width=ref_width,
                height=ref_height,
                resampling=resampling,
                src_nodata=src.nodata,
                nodata=nodata_out,
            ) as vrt:
                data = vrt.read(1)
                out_profile = vrt.profile.copy()

    out_profile.update(driver="GTiff", compress="deflate", predictor=3, tiled=True)

    with rasterio.open(out_path, "w", **out_profile) as dst:
        dst.write(data, 1)

    return out_path, data


## 5. Run the clip for both datasets

In [ ]:
results = {}

for key, cfg in SOURCES.items():
    if DEMO_MODE:
        src_path = DEMO_SOURCE_PATHS[key]
    else:
        src_path = "/vsicurl/" + cfg["url"]

    out_path = os.path.join(OUTPUT_DIR, f"basilicata_{key}_clip.tif")
    print(f"Clipping '{key}' ({cfg['description']}) ...")
    print(f"  source: {src_path}")

    path, arr = clip_to_reference(
        src_path, out_path,
        resampling=cfg["resampling"],
        nodata_out=cfg["nodata_out"],
    )
    results[key] = {"path": path, "array": arr}
    valid = arr[~np.isnan(arr)] if np.isnan(cfg["nodata_out"]) else arr[arr != cfg["nodata_out"]]
    print(f"  -> wrote {path}  (shape={arr.shape}, "
          f"valid min/max = {valid.min():.4g} / {valid.max():.4g})\n")


## 6. Validation

Confirm every clipped output shares the reference raster's CRS, dimensions, transform, and
bounds exactly (this is what "perfect clip" means here — pixel-grid identical, not just an
overlapping bounding box).

In [ ]:
with rasterio.open(REFERENCE_TIF) as ref:
    ref_profile_check = (ref.crs, ref.width, ref.height, ref.transform, ref.bounds)

all_ok = True
for key, res in results.items():
    with rasterio.open(res["path"]) as out:
        check = (out.crs, out.width, out.height, out.transform, out.bounds)
        ok = (
            check[0] == ref_profile_check[0]
            and check[1] == ref_profile_check[1]
            and check[2] == ref_profile_check[2]
            and check[3] == ref_profile_check[3]
        )
        all_ok &= ok
        status = "OK — exact grid match" if ok else "MISMATCH"
        print(f"[{status}] {key}")
        print(f"  CRS      : {check[0]}  (ref: {ref_profile_check[0]})")
        print(f"  size     : {check[1]}x{check[2]}  (ref: {ref_profile_check[1]}x{ref_profile_check[2]})")
        print(f"  bounds   : {check[4]}")
        print(f"  transform matches ref: {check[3] == ref_profile_check[3]}")

assert all_ok, "One or more clipped rasters do not exactly match the reference grid."
print("\nAll clipped rasters are pixel-for-pixel aligned with the reference raster.")


## 7. Quick visual check

Full-resolution arrays here are ~10-14k pixels per side, so the preview reads/downsamples everything to a manageable display size (this is for visual QA only — the saved GeoTIFFs remain full resolution).

In [ ]:
PREVIEW_MAX_DIM = 1500  # max pixels along the longer side, for display only

def preview_array(path, is_ref_categorical=False):
    with rasterio.open(path) as src:
        scale = min(1.0, PREVIEW_MAX_DIM / max(src.width, src.height))
        out_w, out_h = max(1, int(src.width * scale)), max(1, int(src.height * scale))
        resamp = Resampling.nearest if is_ref_categorical else Resampling.average
        arr = src.read(1, out_shape=(out_h, out_w), resampling=resamp).astype("float32")
        nodata = src.nodata
    if nodata is not None and not np.isnan(nodata):
        arr[arr == nodata] = np.nan
    elif is_ref_categorical:
        arr[arr == 0] = np.nan  # treat 0 as background for display only
    return arr

maes_preview = preview_array(REFERENCE_TIF, is_ref_categorical=True)
k_preview = preview_array(results["epic_k_factor"]["path"])
r_preview = preview_array(results["gloresate_erosivity"]["path"])

extent = [ref_bounds.left, ref_bounds.right, ref_bounds.bottom, ref_bounds.top]
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

axes[0].imshow(maes_preview, cmap="tab20", extent=extent)
axes[0].set_title("MAES Level-2 (reference, 2017)")

im1 = axes[1].imshow(k_preview, cmap="YlOrBr", extent=extent)
axes[1].set_title("EPIC K-Factor (clipped)")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(r_preview, cmap="viridis", extent=extent)
axes[2].set_title("GloRESatE erosivity (clipped)")
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

for ax in axes:
    ax.set_xlabel("Easting (EPSG:6875)")
axes[0].set_ylabel("Northing (EPSG:6875)")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "basilicata_clip_preview.png"), dpi=150)
plt.show()


## Notes

- **Resampling**: both source layers are continuous variables (K-factor, erosivity), so
  `Resampling.bilinear` is used when up-sampling them onto the finer 10 m reference grid.
  Switch to `Resampling.cubic` if you want smoother interpolation, or `Resampling.average` if
  you later go the other way (aggregating fine data onto a coarser grid).
- **No download of the full global rasters**: `/vsicurl/` + `WarpedVRT` means GDAL only issues
  HTTP range requests for the bytes covering the Basilicata window — this works because both
  Data Hub layers are served as Cloud-Optimized GeoTIFFs.
- **Combining K-factor × R-factor**: `epic_k_factor` and `gloresate_erosivity` correspond to the
  K and R terms of USLE/RUSLE soil-loss estimation; once aligned to the same grid as the MAES
  Level-2 layer they can be multiplied directly (plus LS/C/P factors) to estimate potential soil
  erosion for Basilicata.
- If you hit network/SSL errors reading the remote COGs in your environment, try setting
  `GDAL_ENV["GDAL_HTTP_UNSAFESSL"] = "YES"` or run once with `DEMO_MODE = True` to confirm the
  rest of the pipeline works before debugging connectivity.
